
# نمایش پایپلاین فاز اول پروژه ترب

این notebook برای **ارائه ویدیویی فاز اول** ساخته شده است. تمام سلول‌ها فقط artifactهای از قبل تولیدشده را می‌خوانند و داده خام، embeddingها یا indexها را تغییر نمی‌دهند.

ترتیب نمایش:

1. معماری pipeline
2. حجم داده و میزان پوشش subsetها
3. Sanity Check
4. EDA
5. Cleaning و ساخت داده یکپارچه
6. Subset Selection
7. Embedding و FAISS
8. Evaluation متنی
9. Image و Multimodal
10. جمع‌بندی نهایی

> جمله پیشنهادی برای ارائه: «این notebook خود pipeline را دوباره اجرا نمی‌کند؛ خروجی checkpointها را می‌خواند و وضعیت کل فاز اول را به شکل قابل ارائه نمایش می‌دهد.»


In [ ]:

from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 100)

def find_phase1_root() -> Path:
    # Find phase1 whether execution starts in the project root or a notebook folder.
    cwd = Path.cwd().resolve()
    candidates = []
    for base in [cwd, *cwd.parents]:
        candidates.extend([base, base / "phase1"])

    for candidate in candidates:
        if (candidate / "torob_rag" / "artifacts").exists():
            return candidate.resolve()

    raise FileNotFoundError(
        "Phase 1 root was not found. Open this notebook from the project workspace."
    )

PHASE1_ROOT = find_phase1_root()
ART = PHASE1_ROOT / "torob_rag" / "artifacts"

def load_json(relative_path: str, required: bool = True):
    path = ART / relative_path
    if not path.exists():
        if required:
            raise FileNotFoundError(f"Missing artifact: {path}")
        return None
    return json.loads(path.read_text(encoding="utf-8"))

def artifact_status(relative_paths):
    rows = []
    for relative_path in relative_paths:
        path = ART / relative_path
        rows.append({
            "artifact": relative_path,
            "status": "READY" if path.exists() else "MISSING",
            "size_kb": round(path.stat().st_size / 1024, 2) if path.exists() else None,
        })
    return pd.DataFrame(rows)

print("Phase 1 root:", PHASE1_ROOT)
print("Artifacts directory:", ART)



## ۱. معماری کلی فاز اول

فاز اول به شکل checkpoint محور طراحی شده است. خروجی هر مرحله در `artifacts` ذخیره می‌شود و مرحله بعدی از آن استفاده می‌کند.

> برای ارائه: «کل داده خام در sanity، EDA و cleaning بررسی می‌شود؛ برای embedding و FAISS به دلیل محدودیت CPU از subset استفاده شده است.»


In [ ]:

stages = [
    "Raw Parquet",
    "Sanity Check",
    "EDA",
    "Cleaning",
    "Subset",
    "Embedding",
    "FAISS",
    "Evaluation",
]

fig, ax = plt.subplots(figsize=(15, 3.2))
ax.set_xlim(-0.5, len(stages) - 0.5)
ax.set_ylim(-0.8, 0.8)
ax.axis("off")

colors = ["#E8EEF7", "#DCEFE8", "#FFF2CC", "#FCE4D6", "#E4DFEC", "#D9EAF7", "#E2F0D9", "#F4CCCC"]
for i, (stage, color) in enumerate(zip(stages, colors)):
    ax.text(
        i, 0, stage, ha="center", va="center", fontsize=11, fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.55", facecolor=color, edgecolor="#444444")
    )
    if i < len(stages) - 1:
        ax.annotate("", xy=(i + 0.67, 0), xytext=(i + 0.33, 0),
                    arrowprops=dict(arrowstyle="->", lw=1.8, color="#444444"))

ax.set_title("Phase 1 Product Retrieval Pipeline", fontsize=15, pad=20)
plt.show()



## ۲. حجم داده و میزان استفاده واقعی

داده خام شامل بیش از یک میلیون Base Product و نزدیک به دو میلیون Member Product است. کل داده در مراحل بررسی و پاک‌سازی استفاده شده، اما embedding متنی و تصویری روی subset اجرا شده‌اند.


In [ ]:

sanity = load_json("checkpoint_01_sanity.json")
cleaning = load_json("clean/cleaning_summary.json")
subset = load_json("subset/subset_v2_summary.json")
image_subset = load_json("images/clip_v1/image_subset_summary.json")
image_download = load_json("images/clip_v1/download_summary.json")

base_total = sanity["profiles"]["base_products"]["rows"]
members_total = sanity["profiles"]["members"]["rows"]
text_subset = subset["subset_size"]
image_selected = image_subset["subset_size"]
image_embedded = image_download["download_ok"]

coverage = pd.DataFrame([
    {"stage": "Raw Base Products", "items": base_total, "percent_of_base": 100.0},
    {"stage": "Text Embedding Subset", "items": text_subset, "percent_of_base": 100 * text_subset / base_total},
    {"stage": "Image Selected", "items": image_selected, "percent_of_base": 100 * image_selected / base_total},
    {"stage": "Images Downloaded/Embedded", "items": image_embedded, "percent_of_base": 100 * image_embedded / base_total},
])
coverage["percent_of_base"] = coverage["percent_of_base"].round(2)
display(coverage)

fig, ax = plt.subplots(figsize=(10, 4.5))
bars = ax.bar(coverage["stage"], coverage["items"], color=["#4C78A8", "#59A14F", "#F28E2B", "#E15759"])
ax.set_title("Dataset Coverage Across Pipeline Stages")
ax.set_ylabel("Number of items")
ax.tick_params(axis="x", rotation=18)
for bar, pct in zip(bars, coverage["percent_of_base"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f"{pct}%", ha="center", va="bottom", fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Total base products: {base_total:,}")
print(f"Total member products: {members_total:,}")
print(f"Text subset: {text_subset:,} ({text_subset/base_total:.1%} of base products)")
print(f"Image subset: {image_selected:,} ({image_selected/base_total:.1%} of base products)")
print(f"Successful images: {image_embedded:,} ({image_embedded/image_selected:.1%} of image subset)")



### متن پیشنهادی برای توضیح حجم داده

«کل دیتاست شامل ۱٬۰۲۲٬۲۹۸ محصول پایه بود. کل داده در sanity، EDA و cleaning بررسی شد. برای embedding متنی به دلیل محدودیت CPU از ۲۰۰ هزار محصول استفاده شد که حدود ۱۹٫۶ درصد کل محصولات است. این subset تصادفی نبود و با وزن‌دهی به impression، view و final click ساخته شد. برای تصویر نیز ۲۰ هزار محصول انتخاب شد و ۱۶٬۲۳۶ تصویر با موفقیت دانلود و embedding شدند.»



## ۳. Sanity Check

Sanity Check قرارداد اولیه داده را بررسی می‌کند: شکل جدول‌ها، missing values، یکتایی کلیدها، روابط کلید خارجی و وضعیت اولیه قیمت.

تفاوت مهم: Sanity می‌پرسد «آیا اسکلت داده سالم است؟» ولی EDA می‌پرسد «داده چه رفتار آماری و توزیعی دارد؟»


In [ ]:

profile_rows = []
for table_name, profile in sanity["profiles"].items():
    profile_rows.append({
        "table": table_name,
        "rows": profile["rows"],
        "columns": profile["cols"],
        "missing_rate": round(profile["missing_rate"], 6),
    })

profiles_df = pd.DataFrame(profile_rows).sort_values("rows", ascending=False)
display(profiles_df)

consistency_df = pd.DataFrame(
    [{"check": key, "value": value} for key, value in sanity["consistency"].items()]
)
display(consistency_df)



## ۴. EDA یا تحلیل اکتشافی داده

EDA برای شناخت توزیع‌ها، outlierها، عدم توازن، missingness و رفتار queryها انجام شده است. نتیجه EDA مستقیماً روی تصمیم‌های cleaning و subset selection اثر گذاشته است.

نکته فنی: برای قیمت فقط میانگین کافی نیست، چون توزیع قیمت long-tail است؛ بنابراین صدک‌هایی مثل p50، p95، p99 و p99.9 بررسی شده‌اند.


In [ ]:

eda = load_json("eda/summary.json")

price = eda["checks"]["price"]
members_per_base = eda["checks"]["members_per_base"]
query_stats = eda["checks"]["search_query"]

eda_summary = pd.DataFrame([
    {"metric": "Price median (p50)", "value": price["p50"]},
    {"metric": "Price p95", "value": price["p95"]},
    {"metric": "Price p99", "value": price["p99"]},
    {"metric": "Price maximum", "value": price["max"]},
    {"metric": "Members per base p50", "value": members_per_base["p50"]},
    {"metric": "Members per base maximum", "value": members_per_base["max"]},
    {"metric": "Query length p50", "value": query_stats["len_p50"]},
    {"metric": "Query length p95", "value": query_stats["len_p95"]},
])
display(eda_summary)

eda_images = [
    ("eda/members_price_log_hist.png", "Price distribution (log scale)"),
    ("eda/members_per_base_hist.png", "Members per base product"),
    ("eda/search_query_len_hist.png", "Search query length"),
    ("eda/missingness_base_products.png", "Base product missingness"),
]

for relative_path, title in eda_images:
    path = ART / relative_path
    display(Markdown(f"### {title}"))
    if path.exists():
        display(Image(filename=str(path), width=760))
    else:
        print("Missing plot:", path)



### نتیجه فنی EDA

- وجود قیمت‌های بسیار بزرگ باعث تعریف سقف قیمت در cleaning شد.
- حجم بیش از یک میلیون محصول باعث استفاده از subset شد.
- کوتاه بودن queryها یکی از دلایل ambiguity و recall پایین baseline متنی است.
- missing بودن تصویر برای برخی محصولات باعث شد image retrieval روی subset تصویردار اجرا شود.



## ۵. Cleaning و داده یکپارچه

در cleaning، نوع کلیدها یکسان می‌شود، قیمت‌های نامعتبر حذف می‌شوند، عنوان برند و دسته join می‌شود، اطلاعات memberها در سطح base aggregate می‌شود و ستون `text_for_embedding` ساخته می‌شود.


In [ ]:

cleaning_table = pd.DataFrame([
    {"metric": "Members before cleaning", "value": cleaning["members_before"]},
    {"metric": "Invalid member rows removed", "value": cleaning["members_invalid_price"]},
    {"metric": "Members after cleaning", "value": cleaning["members_after"]},
    {"metric": "Base product rows", "value": cleaning["bp_rows"]},
    {"metric": "Empty embedding texts", "value": cleaning["bp_text_empty"]},
    {"metric": "Maximum accepted price", "value": cleaning["price_max_cap"]},
])
display(cleaning_table)

removed_pct = 100 * cleaning["members_invalid_price"] / cleaning["members_before"]
print(f"Removed member rows: {removed_pct:.4f}%")
print("All base products received a non-empty text_for_embedding." if cleaning["bp_text_empty"] == 0 else "Some embedding texts are empty.")



## ۶. Subset Selection

Subset متنی با امتیاز رفتار کاربران ساخته شده است:

\[
score = 1 \times impression + 5 \times view + 20 \times final\_click
\]

این روش از نمونه‌گیری تصادفی بهتر است، چون محصولاتی را نگه می‌دارد که در رفتار واقعی کاربران مهم‌تر بوده‌اند.


In [ ]:

subset_table = pd.DataFrame([
    {"item": "Subset version", "value": subset["subset_version"]},
    {"item": "Subset size", "value": subset["subset_size"]},
    {"item": "Impression unique products", "value": subset["impression_unique"]},
    {"item": "View unique products", "value": subset["view_unique"]},
    {"item": "Final-click unique products", "value": subset["final_click_unique"]},
    {"item": "Impression weight", "value": subset["weights"]["impression"]},
    {"item": "View weight", "value": subset["weights"]["view"]},
    {"item": "Final-click weight", "value": subset["weights"]["final_click"]},
])
display(subset_table)



## ۷. Embedding و FAISS

متن هر محصول با مدل چندزبانه به بردار تبدیل می‌شود. سپس بردارها L2-normalize می‌شوند و در `IndexFlatIP` قرار می‌گیرند. برای بردارهای normalize شده، Inner Product معادل Cosine Similarity عمل می‌کند.

مدل متنی: `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`

مدل تصویری: `openai/clip-vit-base-patch32`


In [ ]:

text_meta = load_json("embeddings/text/text_embeddings_meta.json")
image_meta = load_json("embeddings/clip_v1/image/image_embeddings_meta.json")
mm_text_meta = load_json("embeddings/clip_v1/multimodal_text/text_embeddings_meta.json")
mm_corpus = load_json("multimodal/clip_v1/mm_corpus_summary.json")

embedding_summary = pd.DataFrame([
    {
        "branch": "Text-only",
        "model": text_meta["model_name"],
        "items": text_meta["n_items"],
        "dimensions": text_meta["dims"],
        "normalized": text_meta["normalize"],
    },
    {
        "branch": "Image-only",
        "model": image_meta["model_name"],
        "items": image_meta["n_items"],
        "dimensions": image_meta["dims"],
        "normalized": image_meta["normalize"],
    },
    {
        "branch": "Multimodal text branch",
        "model": mm_text_meta["model_name"],
        "items": mm_text_meta["n_items"],
        "dimensions": mm_text_meta["dims"],
        "normalized": mm_text_meta["normalize"],
    },
])
display(embedding_summary)

display(artifact_status([
    "embeddings/text/embeddings.npy",
    "embeddings/text/text_index.parquet",
    "vector_db/faiss/text/faiss_text.index",
    "embeddings/clip_v1/image/embeddings.npy",
    "vector_db/faiss/clip_v1/image/faiss_image.index",
    "multimodal/clip_v1/mm_corpus.parquet",
]))



## ۸. Evaluation متنی

Ground truth از `base_views` ساخته شده است؛ یعنی محصولاتی که کاربر بعد از جست‌وجو دیده، relevant فرض شده‌اند. فقط queryهایی ارزیابی شده‌اند که حداقل یک relevant item داخل index داشته باشند.

- **Recall@k:** آیا حداقل یک جواب relevant در k نتیجه اول آمده است؟
- **MRR@k:** اولین جواب relevant در چه رتبه‌ای آمده است؟
- **NDCG@k:** کیفیت رتبه‌بندی با توجه به جایگاه جواب relevant چقدر است؟


In [ ]:

text_eval = load_json("eval/text/eval_summary.json")
metrics = text_eval["metrics_mean"]

metric_df = pd.DataFrame([
    {"metric": key, "value": value}
    for key, value in metrics.items()
]).sort_values("metric")
display(metric_df)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, prefix, title in zip(axes, ["recall@", "mrr@", "ndcg@"], ["Recall@k", "MRR@k", "NDCG@k"]):
    items = sorted(
        [(int(key.split("@")[1]), value) for key, value in metrics.items() if key.startswith(prefix)],
        key=lambda item: item[0],
    )
    ks = [item[0] for item in items]
    values = [item[1] for item in items]
    ax.plot(ks, values, marker="o", linewidth=2)
    ax.set_xticks(ks)
    ax.set_title(title)
    ax.set_xlabel("k")
    ax.set_ylabel("score")

plt.tight_layout()
plt.show()

print("Evaluation samples:", f"{text_eval['eval_samples']:,}")
print("Index size:", f"{text_eval['index_size']:,}")
print("Failures in top-10:", f"{text_eval['failures_count']:,}")



### تفسیر نتیجه متنی

اعداد baseline متنی پایین هستند، اما این به معنی خراب بودن pipeline نیست. عوامل مهم:

- queryهای کوتاه و مبهم
- پوشش ۲۰۰ هزار محصول از بیش از یک میلیون محصول
- استفاده از مدل embedding سبک
- استفاده از رفتار کاربر به عنوان ground truth
- نبود reranker قوی مثل cross-encoder

هدف فاز اول ساخت baseline قابل توسعه و قابل ارزیابی بوده است.


In [ ]:

failure_path = ART / "eval" / "text" / "failure_examples_readable.csv"
if failure_path.exists():
    failures = pd.read_csv(failure_path)
    columns = [column for column in ["query", "num_relevant_in_index", "top1", "top1_name"] if column in failures.columns]
    display(failures[columns].head(10) if columns else failures.head(10))
else:
    print("Readable failure examples are not available.")



## ۹. Image-only و Multimodal

برای تصویر، ابتدا ۲۰ هزار محصول دارای تصویر انتخاب شدند، سپس تصاویر دانلود و با CLIP embedding شدند. برای multimodal، متن و تصویر جداگانه retrieval می‌شوند و امتیازها با Late Fusion ترکیب می‌شوند:

\[
score = \alpha \times text\_score + (1-\alpha) \times image\_score
\]

ارزیابی‌های image و multimodal در این فاز **sanity evaluation** هستند؛ هدف آن‌ها بررسی صحت mapping و index است، نه سنجش کامل تجربه کاربر.


In [ ]:

image_sanity = load_json("eval/image/clip_v1/sanity_report.json")
mm_sanity = load_json("eval/multimodal/clip_v1/mm_sanity_eval.json")

optional_summary = pd.DataFrame([
    {
        "system": "Image-only (CLIP)",
        "items": image_sanity["n_items"],
        "main_result": f"self_recall@1 = {image_sanity['self_recall@1']:.4f}",
        "evaluation_type": "Self-retrieval sanity",
    },
    {
        "system": "Multimodal (Late Fusion)",
        "items": mm_sanity["sample_n"],
        "main_result": f"recall@10 = {mm_sanity['metrics_mean']['recall@10']:.4f}",
        "evaluation_type": "Self-embedding sanity",
    },
])
display(optional_summary)
print("Multimodal alpha_text:", mm_sanity["alpha_text"])



## ۱۰. جمع‌بندی نهایی برای ارائه

«فاز اول لایه داده و retrieval پروژه است. کل داده خام در sanity، EDA و cleaning بررسی شد. برای embedding متنی به دلیل محدودیت CPU از subset دویست‌هزارتایی، یعنی حدود ۱۹٫۶ درصد کل base productها، استفاده شد. embeddingها در FAISS ذخیره شدند و retrieval با Recall، MRR و NDCG ارزیابی شد. بخش تصویری با CLIP و بخش multimodal با Late Fusion نیز پیاده‌سازی شدند. خروجی‌های این فاز، شامل داده پاک‌سازی‌شده، embeddingها و indexها، در فاز دوم توسط Agent به عنوان RAG tool استفاده می‌شوند.»


In [ ]:

final_status = pd.DataFrame([
    {"component": "Raw data sanity", "status": "READY", "evidence": "checkpoint_01_sanity.json"},
    {"component": "EDA", "status": "READY", "evidence": "eda/summary.json + plots"},
    {"component": "Cleaning", "status": "READY", "evidence": "clean/unified_base_products.parquet"},
    {"component": "Text subset", "status": "READY", "evidence": "subset/base_products_subset_v2.parquet"},
    {"component": "Text embeddings", "status": "READY", "evidence": "embeddings/text/embeddings.npy"},
    {"component": "Text FAISS", "status": "READY", "evidence": "vector_db/faiss/text/faiss_text.index"},
    {"component": "Text evaluation", "status": "READY", "evidence": "eval/text/eval_summary.json"},
    {"component": "Image retrieval", "status": "READY", "evidence": "eval/image/clip_v1/sanity_report.json"},
    {"component": "Multimodal retrieval", "status": "READY", "evidence": "eval/multimodal/clip_v1/mm_sanity_eval.json"},
])
display(final_status)
print("Phase 1 presentation dashboard completed successfully.")
